In [1]:
  !pip install -q "numpy>=1.24,<2" "nibabel>=5,<6" PyYAML tqdm huggingface-hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires num

In [2]:
  DATA = "/content/drive/MyDrive/mrixfields2026/validation_prospective/T1W"
  OUTPUT = "/content/mrixfields_runs/task3_t1w"
  WEIGHTS = "/content/mrixfields_weights"

In [3]:
  !python task3_pipeline.py fetch-official
  !python task3_pipeline.py download-checkpoint --destination "{WEIGHTS}"

Cloning into '/content/official/MRIxFields2026'...
remote: Enumerating objects: 547, done.
remote: Counting objects: 100% (547/547), done.
remote: Compressing objects: 100% (252/252), done.
remote: Total 547 (delta 230), reused 520 (delta 208), pack-reused 0 (from 0)
Receiving objects: 100% (547/547), 9.39 MiB | 21.89 MiB/s, done.
Resolving deltas: 100% (230/230), done.
From https://github.com/MRIxFields/MRIxFields2026
 * branch            f0f4ab2c54ab878b48ddfea8d13f4da512ca411e -> FETCH_HEAD
HEAD is now at f0f4ab2 20260529: Update 1
Official repository ready: /content/official/MRIxFields2026
Revision: f0f4ab2c54ab878b48ddfea8d13f4da512ca411e
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:747: UserWarning: Not enough free disk space to download the file. The expected file size is: 631.35 MB. The target location /content/drive/MyDrive/mrixfields_weights/.cache/huggingface/download/task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights only has 283.43 MB

In [4]:
  !python task3_pipeline.py plan \
    --input-root "{DATA}" \
    --output-dir "{OUTPUT}" \
    --modalities T1W

Input volumes:   15
Modalities:      T1W
Inference cases: 60 (4 targets per input; no identities)
Full Task 3:     no (partial modality set)
T1W 0.1T->1.5T ID 0001 domain=01 -> /content/drive/MyDrive/mrixfields_runs/task3_t1w/task3_0.1T_to_1.5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz
T1W 0.1T->3T ID 0001 domain=02 -> /content/drive/MyDrive/mrixfields_runs/task3_t1w/task3_0.1T_to_3T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz
T1W 0.1T->5T ID 0001 domain=03 -> /content/drive/MyDrive/mrixfields_runs/task3_t1w/task3_0.1T_to_5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz
T1W 0.1T->7T ID 0001 domain=04 -> /content/drive/MyDrive/mrixfields_runs/task3_t1w/task3_0.1T_to_7T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz
T1W 0.1T->1.5T ID 0002 domain=01 -> /content/drive/MyDrive/mrixfields_runs/task3_t1w/task3_0.1T_to_1.5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0002.nii.gz
T1W 0.1T->3T ID 0002 domain=02 -> /content/dri

In [7]:
  !pip install -q lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.8 MB/s eta 0:00:00


In [8]:
  CHECKPOINT = WEIGHTS + "/task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth"

  !python task3_pipeline.py infer \
    --input-root "{DATA}" \
    --output-dir "{OUTPUT}_smoke" \
    --checkpoint "{CHECKPOINT}" \
    --modalities T1W \
    --device cuda \
    --batch-size 1 \
    --smoke

Loading one shared checkpoint on cuda ...
Running 1 case(s); batch size 1
[1/1] T1W 0.1T->1.5T ID 0001, joint domain 1
Inference complete. Manifest: /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_run_manifest.json


In [9]:
  !python task3_pipeline.py validate-outputs \
    --input-root "{DATA}" \
    --output-dir "{OUTPUT}_smoke" \
    --modalities T1W \
    --smoke


[001/001] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_1.5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz: OK
Validated 1 outputs. Report: /content/runs/output_validation.json


In [10]:
!python task3_pipeline.py

usage: task3_pipeline.py [-h]
                         {fetch-official,download-checkpoint,audit,plan,infer,validate-outputs}
                         ...
task3_pipeline.py: error: the following arguments are required: command


In [11]:
FULL_OUTPUT = OUTPUT + "_smoke"

!python task3_pipeline.py infer \
  --input-root "{DATA}" \
  --output-dir "{FULL_OUTPUT}" \
  --checkpoint "{CHECKPOINT}" \
  --modalities T1W \
  --device cuda \
  --batch-size 1 \
  --resume

Loading one shared checkpoint on cuda ...
Running 60 case(s); batch size 1
[1/60] resume: /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_1.5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz
[2/60] T1W 0.1T->3T ID 0001, joint domain 2
[3/60] T1W 0.1T->5T ID 0001, joint domain 3
[4/60] T1W 0.1T->7T ID 0001, joint domain 4
[5/60] T1W 0.1T->1.5T ID 0002, joint domain 1
[6/60] T1W 0.1T->3T ID 0002, joint domain 2
[7/60] T1W 0.1T->5T ID 0002, joint domain 3
[8/60] T1W 0.1T->7T ID 0002, joint domain 4
[9/60] T1W 0.1T->1.5T ID 0003, joint domain 1
[10/60] T1W 0.1T->3T ID 0003, joint domain 2
[11/60] T1W 0.1T->5T ID 0003, joint domain 3
[12/60] T1W 0.1T->7T ID 0003, joint domain 4
[13/60] T1W 1.5T->0.1T ID 0004, joint domain 0
[14/60] T1W 1.5T->3T ID 0004, joint domain 2
[15/60] T1W 1.5T->5T ID 0004, joint domain 3
[16/60] T1W 1.5T->7T ID 0004, joint domain 4
[17/60] T1W 1.5T->0.1T ID 0005, joint domain 0
[18/60] T1W 1.5T->3T ID 0005, joint domain 2
[19/60] T1

In [12]:
!python task3_pipeline.py validate-outputs \
  --input-root "{DATA}" \
  --output-dir "{FULL_OUTPUT}" \
  --modalities T1W \
  --report "{FULL_OUTPUT}/t1w_validation.json"

[001/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_1.5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz: OK
[002/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_3T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz: OK
[003/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz: OK
[004/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_7T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0001.nii.gz: OK
[005/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_1.5T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0002.nii.gz: OK
[006/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_3T_T1W/stargan_v2/pro_pretrained/epoch50/P_T1W_0.1T_0002.nii.gz: OK
[007/060] /content/drive/MyDrive/mrixfields_runs/task3_t1w_smoke/task3_0.1T_to_5T_T1W/stargan_v2/pro_pretrained/ep